<a href="https://colab.research.google.com/github/MohmmadSami/PDC-Lab-task/blob/main/53199_PDC_Lab_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Lab Task 08: Introduction to CUDA

---

| | |
|---|---|
| **Student Name** | Mohammad Sami |
| **SAP ID** | 53199 |
| **Section** | CS-6-1 |
| **Course** | PDC (Parallel & Distributed Computing) |
| **Lab Instructor** | Muhammad Nadeem Khan |
| **Platform** | Google Colab — T4 GPU |

---

## ⚙️ Setup Instructions
**Before running anything:**
1. Click **Runtime** (top menu)
2. Click **Change runtime type**
3. Select **T4 GPU**
4. Click **Save**
5. Then run cells from top to bottom ▶️

---

## ✅ Step 1: Verify GPU is Available

In [1]:
# Check that a GPU is connected
!nvidia-smi

Mon Mar 16 09:41:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Check CUDA compiler version
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


---
## 📝 Task 1: Add Two Arrays on GPU vs CPU

**Objective:** Write a CUDA program to add two arrays on the GPU and compare the execution time with a CPU implementation.

**What this program does:**
- Creates two arrays of **1,000,000** integers
- Adds them element-wise using a **CUDA kernel** on the GPU (parallel)
- Adds them element-wise using a **CPU function** (sequential)
- Compares results and prints execution times

In [3]:
%%writefile task1_array_addition.cu
/*
 * Lab 7 - Task 1
 * CUDA Program: Add Two Arrays on GPU and Compare Execution Time with CPU
 * Compile: nvcc task1_array_addition.cu -o task1
 * Run:     ./task1
 */

#include <iostream>
#include <ctime>

#define N 1000000  // Array size: 1 million elements

// ─────────────────────────────────────────────────────────────
//  GPU KERNEL: each thread adds one pair of elements
// ─────────────────────────────────────────────────────────────
__global__ void addArraysGPU(int* a, int* b, int* c, int n) {
    // Compute a globally unique thread index
    int idx = blockIdx.x * blockDim.x + threadIdx.x;

    // Guard: make sure we don't go out of bounds
    if (idx < n) {
        c[idx] = a[idx] + b[idx];
    }
}

// ─────────────────────────────────────────────────────────────
//  CPU FUNCTION: sequential element-wise addition
// ─────────────────────────────────────────────────────────────
void addArraysCPU(int* a, int* b, int* c, int n) {
    for (int i = 0; i < n; i++) {
        c[i] = a[i] + b[i];
    }
}

int main() {
    // 1. Allocate and initialize host (CPU) arrays
    int* h_a     = new int[N];
    int* h_b     = new int[N];
    int* h_c_gpu = new int[N];
    int* h_c_cpu = new int[N];

    for (int i = 0; i < N; i++) {
        h_a[i] = i;       // a = [0, 1, 2, ..., N-1]
        h_b[i] = N - i;   // b = [N, N-1, ..., 1]
    }
    // Expected: every element of result = N = 1,000,000

    // 2. Allocate device (GPU) memory
    int *d_a, *d_b, *d_c;
    cudaMalloc((void**)&d_a, N * sizeof(int));
    cudaMalloc((void**)&d_b, N * sizeof(int));
    cudaMalloc((void**)&d_c, N * sizeof(int));

    // 3. Copy input arrays CPU -> GPU
    cudaMemcpy(d_a, h_a, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, N * sizeof(int), cudaMemcpyHostToDevice);

    // 4. Define grid/block dimensions
    int threadsPerBlock = 256;
    int blocksPerGrid   = (N + threadsPerBlock - 1) / threadsPerBlock;

    // 5. GPU Timing using CUDA Events
    cudaEvent_t gpuStart, gpuStop;
    cudaEventCreate(&gpuStart);
    cudaEventCreate(&gpuStop);

    cudaEventRecord(gpuStart);
    addArraysGPU<<<blocksPerGrid, threadsPerBlock>>>(d_a, d_b, d_c, N);
    cudaEventRecord(gpuStop);
    cudaEventSynchronize(gpuStop);

    float gpuMs = 0;
    cudaEventElapsedTime(&gpuMs, gpuStart, gpuStop);

    // 6. Copy result GPU -> CPU
    cudaMemcpy(h_c_gpu, d_c, N * sizeof(int), cudaMemcpyDeviceToHost);

    // 7. CPU timing
    clock_t cpuStart = clock();
    addArraysCPU(h_a, h_b, h_c_cpu, N);
    clock_t cpuEnd   = clock();
    double cpuMs = (double)(cpuEnd - cpuStart) / CLOCKS_PER_SEC * 1000.0;

    // 8. Verify results match
    bool match = true;
    for (int i = 0; i < N; i++) {
        if (h_c_gpu[i] != h_c_cpu[i]) { match = false; break; }
    }

    // 9. Print results
    std::cout << "===== Task 1: Array Addition (N = " << N << ") =====\n";
    std::cout << "First 5 GPU results : ";
    for (int i = 0; i < 5; i++) std::cout << h_c_gpu[i] << " ";
    std::cout << "\nFirst 5 CPU results : ";
    for (int i = 0; i < 5; i++) std::cout << h_c_cpu[i] << " ";
    std::cout << "\n";
    std::cout << "Results Match       : " << (match ? "YES ✓" : "NO ✗") << "\n";
    std::cout << "GPU Execution Time  : " << gpuMs  << " ms\n";
    std::cout << "CPU Execution Time  : " << cpuMs  << " ms\n";
    std::cout << "Speedup (CPU/GPU)   : " << cpuMs / gpuMs << "x\n";

    // 10. Free memory
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    cudaEventDestroy(gpuStart); cudaEventDestroy(gpuStop);
    delete[] h_a; delete[] h_b; delete[] h_c_gpu; delete[] h_c_cpu;

    return 0;
}

Writing task1_array_addition.cu


In [4]:
# Compile Task 1
!nvcc task1_array_addition.cu -o task1
print("✅ Task 1 compiled successfully!")

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
✅ Task 1 compiled successfully!


In [5]:
# Run Task 1
!./task1

===== Task 1: Array Addition (N = 1000000) =====
First 5 GPU results : 1000000 1000000 1000000 1000000 1000000 
First 5 CPU results : 1000000 1000000 1000000 1000000 1000000 
Results Match       : YES ✓
GPU Execution Time  : 106.57 ms
CPU Execution Time  : 5.635 ms
Speedup (CPU/GPU)   : 0.0528761x


---
## 📝 Task 2: Square Each Element in an Array

**Objective:** Implement a CUDA kernel to compute the square of each element in an array and print the results.

**What this program does:**
- Creates an input array **[1, 2, 3, ..., 20]**
- Uses a **CUDA kernel** to square each element in parallel on GPU
- Also computes squares on **CPU** for verification
- Prints both results and confirms they match

In [6]:
%%writefile task2_square_elements.cu
/*
 * Lab 7 - Task 2
 * CUDA Program: Compute the Square of Each Element in an Array
 * Compile: nvcc task2_square_elements.cu -o task2
 * Run:     ./task2
 */

#include <iostream>
#include <ctime>

#define N 20  // Small size so all results are easy to print

// ─────────────────────────────────────────────────────────────
//  GPU KERNEL: each thread squares one element
// ─────────────────────────────────────────────────────────────
__global__ void squareArrayGPU(int* input, int* output, int n) {
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        output[idx] = input[idx] * input[idx];
    }
}

// ─────────────────────────────────────────────────────────────
//  CPU FUNCTION: sequential squaring
// ─────────────────────────────────────────────────────────────
void squareArrayCPU(int* input, int* output, int n) {
    for (int i = 0; i < n; i++) {
        output[i] = input[i] * input[i];
    }
}

int main() {
    // 1. Allocate and initialize host arrays
    int* h_input      = new int[N];
    int* h_output_gpu = new int[N];
    int* h_output_cpu = new int[N];

    std::cout << "===== Task 2: Squaring Array Elements (N = " << N << ") =====\n";
    std::cout << "Input array  : ";
    for (int i = 0; i < N; i++) {
        h_input[i] = i + 1;  // values 1 to 20
        std::cout << h_input[i] << " ";
    }
    std::cout << "\n";

    // 2. Allocate device memory
    int *d_input, *d_output;
    cudaMalloc((void**)&d_input,  N * sizeof(int));
    cudaMalloc((void**)&d_output, N * sizeof(int));

    // 3. Copy input CPU -> GPU
    cudaMemcpy(d_input, h_input, N * sizeof(int), cudaMemcpyHostToDevice);

    // 4. Grid/block dimensions
    int threadsPerBlock = 256;
    int blocksPerGrid   = (N + threadsPerBlock - 1) / threadsPerBlock;

    // 5. GPU Timing
    cudaEvent_t gpuStart, gpuStop;
    cudaEventCreate(&gpuStart);
    cudaEventCreate(&gpuStop);

    cudaEventRecord(gpuStart);
    squareArrayGPU<<<blocksPerGrid, threadsPerBlock>>>(d_input, d_output, N);
    cudaEventRecord(gpuStop);
    cudaEventSynchronize(gpuStop);

    float gpuMs = 0;
    cudaEventElapsedTime(&gpuMs, gpuStart, gpuStop);

    // 6. Copy result GPU -> CPU
    cudaMemcpy(h_output_gpu, d_output, N * sizeof(int), cudaMemcpyDeviceToHost);

    // 7. CPU computation
    clock_t cpuStart = clock();
    squareArrayCPU(h_input, h_output_cpu, N);
    clock_t cpuEnd   = clock();
    double cpuMs = (double)(cpuEnd - cpuStart) / CLOCKS_PER_SEC * 1000.0;

    // 8. Verify
    bool match = true;
    for (int i = 0; i < N; i++) {
        if (h_output_gpu[i] != h_output_cpu[i]) { match = false; break; }
    }

    // 9. Print results
    std::cout << "GPU squares  : ";
    for (int i = 0; i < N; i++) std::cout << h_output_gpu[i] << " ";
    std::cout << "\n";
    std::cout << "CPU squares  : ";
    for (int i = 0; i < N; i++) std::cout << h_output_cpu[i] << " ";
    std::cout << "\n";
    std::cout << "Results Match       : " << (match ? "YES ✓" : "NO ✗") << "\n";
    std::cout << "GPU Execution Time  : " << gpuMs  << " ms\n";
    std::cout << "CPU Execution Time  : " << cpuMs  << " ms\n";

    // 10. Free memory
    cudaFree(d_input); cudaFree(d_output);
    cudaEventDestroy(gpuStart); cudaEventDestroy(gpuStop);
    delete[] h_input; delete[] h_output_gpu; delete[] h_output_cpu;

    return 0;
}

Writing task2_square_elements.cu


In [7]:
# Compile Task 2
!nvcc task2_square_elements.cu -o task2
print("✅ Task 2 compiled successfully!")

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
✅ Task 2 compiled successfully!


In [8]:
# Run Task 2
!./task2

===== Task 2: Squaring Array Elements (N = 20) =====
Input array  : 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 
GPU squares  : 1 4 9 16 25 36 49 64 81 100 121 144 169 196 225 256 289 324 361 400 
CPU squares  : 1 4 9 16 25 36 49 64 81 100 121 144 169 196 225 256 289 324 361 400 
Results Match       : YES ✓
GPU Execution Time  : 11.716 ms
CPU Execution Time  : 0.002 ms


---
## 📸 Save Your Output (for Lab Report)
Run the cell below to print both outputs together — copy this for your lab report deliverable.

In [9]:
print("=" * 55)
print("  LAB TASK 08 — CUDA RESULTS")
print("  Student : Mohammad Sami")
print("  SAP ID  : 53199")
print("  Section : CS-6-1")
print("  Course  : PDC")
print("=" * 55)
print()
print("--- TASK 1 OUTPUT ---")
!./task1
print()
print("--- TASK 2 OUTPUT ---")
!./task2

  LAB TASK 08 — CUDA RESULTS
  Student : Mohammad Sami
  SAP ID  : 53199
  Section : CS-6-1
  Course  : PDC

--- TASK 1 OUTPUT ---
===== Task 1: Array Addition (N = 1000000) =====
First 5 GPU results : 1000000 1000000 1000000 1000000 1000000 
First 5 CPU results : 1000000 1000000 1000000 1000000 1000000 
Results Match       : YES ✓
GPU Execution Time  : 0.194688 ms
CPU Execution Time  : 4.9 ms
Speedup (CPU/GPU)   : 25.1685x

--- TASK 2 OUTPUT ---
===== Task 2: Squaring Array Elements (N = 20) =====
Input array  : 1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 
GPU squares  : 1 4 9 16 25 36 49 64 81 100 121 144 169 196 225 256 289 324 361 400 
CPU squares  : 1 4 9 16 25 36 49 64 81 100 121 144 169 196 225 256 289 324 361 400 
Results Match       : YES ✓
GPU Execution Time  : 0.146784 ms
CPU Execution Time  : 0.001 ms
